In [ ]:
# vae_FC_Only_Annealing_128.pthの検証用コード
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score
import random
from skimage.metrics import structural_similarity as ssim

# 文字化け対策の強制適用
import japanize_matplotlib
japanize_matplotlib.japanize()

# =====================================================================
# 1. 前処理とモデルのインポート準備 (FC_Only版)
# =====================================================================
transform = transforms.Compose([
    transforms.CenterCrop(820),
    transforms.Resize((128, 128), interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

# 評価対象のモデルクラス定義 (VAE_FC)
class VAE_FC(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_FC, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(16384, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            nn.Linear(1024, 16384), nn.Sigmoid()
        )

    def forward(self, x):
        h = torch.flatten(x, start_dim=1)
        h = self.encoder(h)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        recon = self.decoder(z)
        return recon.view(-1, 1, 128, 128), mu, logvar

# =====================================================================
# 2. 全体評価と可視化の関数群 (SSIM Top-K仕様)
# =====================================================================
def evaluate_dataset(model, test_dir, device, threshold, top_k=130):
    """データセット全体を推論し、評価指標、スコア、ファイルパスを返す"""
    model.eval()
    y_true, y_pred, scores, all_image_paths = [], [], [], []
    
    test_dir = Path(test_dir)
    print(f"テストデータ全件の評価を開始します... (Top-K: {top_k})")
    
    with torch.no_grad():
        for folder in test_dir.iterdir():
            if not folder.is_dir():
                continue
                
            true_label = 0 if folder.name == "good" else 1
            
            for img_path in folder.glob("*.png"):
                img = Image.open(img_path).convert('RGB')
                input_tensor = transform(img).unsqueeze(0).to(device)
                recon_tensor, _, _ = model(input_tensor)
                
                input_img = input_tensor.squeeze().cpu().numpy()
                recon_img = recon_tensor.squeeze().cpu().numpy()
                
                # SSIMの計算
                _, ssim_map = ssim(input_img, recon_img, data_range=1.0, win_size=11, full=True)
                anomaly_map = 1.0 - ssim_map
                
                # Top-Kの平均をスコアとして算出
                sorted_anomaly = np.sort(anomaly_map.flatten())[::-1]
                top_k_score = np.mean(sorted_anomaly[:top_k])
                
                pred_label = 1 if top_k_score > threshold else 0
                
                y_true.append(true_label)
                y_pred.append(pred_label)
                scores.append(top_k_score)
                all_image_paths.append(img_path)
                
    return np.array(y_true), np.array(y_pred), np.array(scores), all_image_paths

def plot_confusion_matrix(y_true, y_pred, threshold, top_k=130):
    """混同行列と正解率・再現率を描画する"""
    acc = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    
    print(f"\n--- 定量評価結果 ---")
    print(f"閾値: {threshold}")
    print(f"正解率 (Accuracy): {acc * 100:.2f}%")
    print(f"再現率 (Recall)  : {recall * 100:.2f}%\n")
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['正常予測', '異常予測'],
                yticklabels=['実際の正常', '実際の異常'],
                annot_kws={"size": 16})
    
    plt.title(f"混同行列 (SSIM Top-{top_k} 閾値={threshold})\nAccuracy: {acc*100:.1f}% | Recall: {recall*100:.1f}%", fontsize=16)
    plt.xlabel('AIの予測', fontsize=14)
    plt.ylabel('実際の正解', fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_score_distribution(y_true, scores, threshold, top_k=130):
    """正常画像と異常画像のスコア分布を描画する"""
    plt.figure(figsize=(10, 6))
    
    normal_scores = scores[y_true == 0]
    anomaly_scores = scores[y_true == 1]
    
    min_score = np.min(scores)
    max_score = np.max(scores)
    common_bins = np.linspace(min_score, max_score, 40)
    
    sns.histplot(normal_scores, bins=common_bins, color='green', alpha=0.5, label='正常 (Good)')
    sns.histplot(anomaly_scores, bins=common_bins, color='red', alpha=0.5, label='異常 (Anomaly)')
    
    plt.axvline(x=threshold, color='blue', linestyle='--', linewidth=2, label=f'現在の閾値 ({threshold})')
    
    plt.title(f"SSIM Top-{top_k}抽出 異常スコアの分布", fontsize=16, fontweight='bold')
    plt.xlabel(f"異常スコア (上位{top_k}パッチの平均)", fontsize=14)
    plt.ylabel("画像の数 (Count)", fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def plot_single_anomaly(model, image_path, device, threshold, top_k=130):
    """1枚の画像を3パネルで可視化する"""
    model.eval()
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        recon_tensor, _, _ = model(input_tensor)
        
    input_img = input_tensor.squeeze().cpu().numpy()
    recon_img = recon_tensor.squeeze().cpu().numpy()
    
    _, ssim_map = ssim(input_img, recon_img, data_range=1.0, win_size=11, full=True)
    anomaly_map = 1.0 - ssim_map
    
    sorted_anomaly = np.sort(anomaly_map.flatten())[::-1]
    top_k_score = np.mean(sorted_anomaly[:top_k])
    
    is_anomaly = top_k_score > threshold
    judgment = "【異常 (Anomaly)】" if is_anomaly else "【正常 (Good)】"

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"[{image_path.parent.name}] {image_path.name} | SSIM Top-{top_k}異常スコア: {top_k_score:.4f} -> 判定: {judgment}", 
                 fontsize=16, fontweight='bold', color='red' if is_anomaly else 'green')
    
    axes[0].imshow(input_img, cmap='gray')
    axes[0].set_title("① 入力画像", fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(recon_img, cmap='gray')
    axes[1].set_title("② VAE復元画像", fontsize=14)
    axes[1].axis('off')
    
    vmax_val = np.max(anomaly_map) if np.max(anomaly_map) > 0.1 else 0.1
    im3 = axes[2].imshow(anomaly_map, cmap='jet', vmin=0, vmax=vmax_val)
    axes[2].set_title(f"③ SSIM異常度マップ\n(赤色が構造の破壊を示す)", fontsize=14)
    axes[2].axis('off')
    fig.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()

# =====================================================================
# 3. 実行ブロック (統合処理)
# =====================================================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")
    
    # -------------------------------------------------------------
    # ★ ここで検証したいモデルと重みファイルを指定します ★
    # -------------------------------------------------------------
    model = VAE_FC(latent_dim=64).to(device)
    
    # ※学習コードに合わせて重みファイル名を指定
    MODEL_PATH = "saved_models/vae_FC_Only_Annealing_128.pth"
    
    # ファイル名が異なる場合は、実際のファイル名に合わせて書き換えてください。
    # 例： MODEL_PATH = "saved_models/vae_FC_Only_Annealing_ep100_beta50_128.pth"
    
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(f"モデルの重みを読み込みました: {MODEL_PATH}\n")
    
    # OS依存のない相対パス
    TEST_DIR = "data/bottle/test"
    
    TOP_K = 130
    
    # ★ ヒストグラムを確認し、緑と赤の山が分かれる最適な値に調整してください
    THRESHOLD = 0.5 
    
    # 1. データセット全件評価
    y_true, y_pred, scores, all_image_paths = evaluate_dataset(model, TEST_DIR, device, THRESHOLD, top_k=TOP_K)
    
    # 2. 混同行列の描画
    plot_confusion_matrix(y_true, y_pred, THRESHOLD, top_k=TOP_K)
    
    # 3. スコア分布の描画
    plot_score_distribution(y_true, scores, THRESHOLD, top_k=TOP_K)
    
    # 4. パターン別（TN, FP, FN, TP）の抽出と可視化
    print("\n--- パターン別の抽出と可視化を開始します ---")
    
    tn_paths, fp_paths, fn_paths, tp_paths = [], [], [], []
    
    for true_lbl, pred_lbl, path in zip(y_true, y_pred, all_image_paths):
        if true_lbl == 0 and pred_lbl == 0:
            tn_paths.append(path)
        elif true_lbl == 0 and pred_lbl == 1:
            fp_paths.append(path)
        elif true_lbl == 1 and pred_lbl == 0:
            fn_paths.append(path)
        elif true_lbl == 1 and pred_lbl == 1:
            tp_paths.append(path)
            
    categories = {
        "【1. 正常を「正常」と正しく判断 (True Negative)】": tn_paths,
        "【2. 正常を「異常」と誤って判断 (False Positive: 過剰検知)】": fp_paths,
        "【3. 異常を「正常」と誤って判断 (False Negative: 見逃し)】": fn_paths,
        "【4. 異常を「異常」と正しく判断 (True Positive)】": tp_paths
    }
    
    for title, paths in categories.items():
        sample_size = min(2, len(paths))
        print(f"\n{title} : 該当 {len(paths)} 件中 {sample_size} 件を表示します。")
        
        if sample_size > 0:
            sampled_paths = random.sample(paths, sample_size)
            for img_path in sampled_paths:
                plot_single_anomaly(model, img_path, device, THRESHOLD, top_k=TOP_K)
        else:
            print("  -> このパターンに該当する画像はありませんでした。")

使用デバイス: cpu


FileNotFoundError: [Errno 2] No such file or directory: 'saved_models/vae_FC_Only_Annealing_128.pth'

In [ ]:
# =====================================================================
# 検証用コード: 全結合層のみのモデル（FC_Only）100epアニーリング版 専用
# 異常スコア: SSIM Top-K (上位130パッチの平均誤差)
# =====================================================================
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score
import random
from skimage.metrics import structural_similarity as ssim

# 文字化け対策の強制適用
import japanize_matplotlib
japanize_matplotlib.japanize()

# =====================================================================
# 1. 前処理とモデルの定義 (FC_Only版)
# =====================================================================
# 学習時と同じ前処理を定義
transform = transforms.Compose([
    transforms.CenterCrop(820),
    transforms.Resize((128, 128), interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

# 検証対象のモデルクラス定義 (学習コードからコピー)
class VAE_FC(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_FC, self).__init__()
        # エンコーダ: 段階的に次元を圧縮 (16384 -> 1024 -> 256 -> 128)
        self.encoder = nn.Sequential(
            nn.Linear(16384, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )
        # 潜在空間へのマッピング
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # デコーダ: 段階的に次元を復元 (64 -> 128 -> 256 -> 1024 -> 16384)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            nn.Linear(1024, 16384), nn.Sigmoid() # 最後にSigmoidで0~1へ
        )

    def forward(self, x):
        # 画像を1次元に平坦化 (batch, 1, 128, 128) -> (batch, 16384)
        h = torch.flatten(x, start_dim=1)
        h = self.encoder(h)
        
        # 潜在変数のサンプリング
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        
        # 復元
        recon = self.decoder(z)
        # 画像の形状に戻す -> (batch, 1, 128, 128)
        return recon.view(-1, 1, 128, 128), mu, logvar

# =====================================================================
# 2. 評価と可視化の関数群 (SSIM Top-K仕様)
# =====================================================================
# ※ これらの関数はモデルアーキテクチャに依存しないため、Shallow3版と同じです。

def evaluate_dataset(model, test_dir, device, threshold, top_k=130):
    """データセット全体を推論し、評価指標、スコア、ファイルパスを返す"""
    model.eval()
    y_true, y_pred, scores, all_image_paths = [], [], [], []
    
    test_dir = Path(test_dir)
    # テストフォルダが存在するか確認
    if not test_dir.exists():
        raise FileNotFoundError(f"テストディレクトリが見つかりません: {test_dir}\n研究室PCのデータパスを確認してください。")

    print(f"テストデータ全件の評価を開始します... (モデル: FC_Only, Top-K: {top_k})")
    
    with torch.no_grad():
        # test/good と test/anomaly フォルダを巡回
        for folder in test_dir.iterdir():
            if not folder.is_dir():
                continue
                
            # 正解ラベルの設定 (good=0, 正常以外=1)
            true_label = 0 if folder.name == "good" else 1
            
            # フォルダ内のPNG画像を取得
            img_paths = list(folder.glob("*.png"))
            if not img_paths:
                continue

            for img_path in img_paths:
                # 画像の読み込みと前処理
                img = Image.open(img_path).convert('RGB')
                input_tensor = transform(img).unsqueeze(0).to(device)
                
                # VAEに入力して復元画像を取得
                recon_tensor, _, _ = model(input_tensor)
                
                # Tensorからnumpy配列(画像)に変換 (128, 128)
                input_img = input_tensor.squeeze().cpu().numpy()
                recon_img = recon_tensor.squeeze().cpu().numpy()
                
                # --- SSIMによる異常スコア計算 ---
                # full=True で画像全体のSSIMマップを取得
                _, ssim_map = ssim(input_img, recon_img, data_range=1.0, win_size=11, full=True)
                # SSIMは1に近いほど正常なので、1から引いて「異常度マップ」にする
                anomaly_map = 1.0 - ssim_map
                
                # 異常度マップを1次元に平坦化し、降順(大きい順)にソート
                sorted_anomaly = np.sort(anomaly_map.flatten())[::-1]
                # 上位K個(Top-K)の平均を、この画像の「異常スコア」とする
                top_k_score = np.mean(sorted_anomaly[:top_k])
                
                # 閾値判定
                pred_label = 1 if top_k_score > threshold else 0
                
                # 結果を記録
                y_true.append(true_label)
                y_pred.append(pred_label)
                scores.append(top_k_score)
                all_image_paths.append(img_path)
                
    print("評価が完了しました。")
    return np.array(y_true), np.array(y_pred), np.array(scores), all_image_paths

def plot_confusion_matrix(y_true, y_pred, threshold, top_k=130):
    """混同行列と定量評価指標(正解率・再現率)を描画する"""
    acc = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    
    print(f"\n--- 定量評価結果 (FC_Only) ---")
    print(f"設定閾値: {threshold}")
    print(f"正解率 (Accuracy): {acc * 100:.2f}% (正常と異常を正しく当てた割合)")
    print(f"再現率 (Recall)  : {recall * 100:.2f}% (実際の異常のうち、異常と判定できた割合)\n")
    
    # 混同行列の計算
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    # ヒートマップとして描画
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['正常予測', '異常予測'],
                yticklabels=['実際の正常', '実際の異常'],
                annot_kws={"size": 16})
    
    plt.title(f"混同行列 (FC_Only | SSIM Top-{top_k})\nAcc: {acc*100:.1f}% | Recall: {recall*100:.1f}%", fontsize=16)
    plt.xlabel('AIの予測', fontsize=14)
    plt.ylabel('実際の正解', fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_score_distribution(y_true, scores, threshold, top_k=130):
    """正常画像と異常画像の異常スコア分布(ヒストグラム)を描画する"""
    plt.figure(figsize=(10, 6))
    
    # 正常と異常でスコアを分ける
    normal_scores = scores[y_true == 0]
    anomaly_scores = scores[y_true == 1]
    
    # 描画の範囲を自動設定
    min_score = np.min(scores)
    max_score = np.max(scores)
    # 棒の幅(bins)を共通にする
    common_bins = np.linspace(min_score, max_score, 40)
    
    # ヒストグラムを描画
    sns.histplot(normal_scores, bins=common_bins, color='green', alpha=0.5, label='正常 (Good)', kde=False)
    sns.histplot(anomaly_scores, bins=common_bins, color='red', alpha=0.5, label='異常 (Anomaly)', kde=False)
    
    # 閾値を青い破線で描画
    plt.axvline(x=threshold, color='blue', linestyle='--', linewidth=2, label=f'現在の閾値 ({threshold})')
    
    plt.title(f"異常スコアの分布 (FC_Only | SSIM Top-{top_k})", fontsize=16, fontweight='bold')
    plt.xlabel(f"異常スコア (上位{top_k}パッチの類似度誤差平均)", fontsize=14)
    plt.ylabel("画像の数 (Count)", fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def plot_single_anomaly(model, image_path, device, threshold, top_k=130):
    """1枚の画像を「入力」「復元」「異常度マップ」の3パネルで可視化する"""
    model.eval()
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        recon_tensor, _, _ = model(input_tensor)
        
    input_img = input_tensor.squeeze().cpu().numpy()
    recon_img = recon_tensor.squeeze().cpu().numpy()
    
    # SSIM計算と異常スコア算出
    _, ssim_map = ssim(input_img, recon_img, data_range=1.0, win_size=11, full=True)
    anomaly_map = 1.0 - ssim_map
    sorted_anomaly = np.sort(anomaly_map.flatten())[::-1]
    top_k_score = np.mean(sorted_anomaly[:top_k])
    
    # 判定結果
    is_anomaly = top_k_score > threshold
    judgment = "【異常 (Anomaly)】" if is_anomaly else "【正常 (Good)】"
    result_color = 'red' if is_anomaly else 'green'

    # 3パネルの描画設定
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    
    # メインタイトル設定 (ファイル名、スコア、判定結果)
    # folder_name = image_path.parent.name # フォルダ名(good, brokenなど)を取得
    fig.suptitle(f"ファイル: {image_path.name} (実際の正解: {image_path.parent.name})\nSSIM Top-{top_k}スコア: {top_k_score:.4f} -> 判定: {judgment}", 
                 fontsize=16, fontweight='bold', color=result_color)
    
    # ① 入力画像
    axes[0].imshow(input_img, cmap='gray')
    axes[0].set_title("① 入力画像 (Gray)", fontsize=14)
    axes[0].axis('off')
    
    # ② VAE復元画像
    axes[1].imshow(recon_img, cmap='gray')
    axes[1].set_title("② VAE復元画像", fontsize=14)
    axes[1].axis('off')
    
    # ③ SSIM異常度マップ (熱マップ)
    # 色の範囲(vmax)を自動調整。最大値が低すぎる場合は0.1を上限にする
    vmax_val = np.max(anomaly_map) if np.max(anomaly_map) > 0.1 else 0.1
    im3 = axes[2].imshow(anomaly_map, cmap='jet', vmin=0, vmax=vmax_val)
    axes[2].set_title(f"③ 異常度マップ (1-SSIM)\n(赤色: 構造が復元できていない箇所)", fontsize=14)
    axes[2].axis('off')
    # カラーバーを追加
    fig.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96]) # タイトルと被らないように調整
    plt.show()

# =====================================================================
# 3. 実行ブロック (研究室PCでの実行用)
# =====================================================================
if __name__ == "__main__":
    # デバイスの設定 (研究室PCにGPUがあればcudaが使われます)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")
    
    # -------------------------------------------------------------
    # ★ 重要: モデルと重みファイルの指定 ★
    # -------------------------------------------------------------
    # 評価するモデルのインスタンスを作成
    model = VAE_FC(latent_dim=64).to(device)
    
    # 【研究室PC用設定】学習コードで保存した重みファイルのパスを指定
    # MODEL_NAME = "FC_Only_Annealing_ep100_beta50" の場合のデフォルト名
    MODEL_PATH = "saved_models/vae_FC_Only_Annealing_ep100_beta50_128.pth"
    
    # ファイルが存在するか確認
    if not Path(MODEL_PATH).exists():
        raise FileNotFoundError(f"重みファイルが見つかりません: {MODEL_PATH}\n学習が完了しているか、パスが正しいか確認してください。")
    
    # 重みをロード
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(f"FC_Only (100ep版) の重みを読み込みました。\n")
    
    # -------------------------------------------------------------
    # ★ 重要: テストデータのパス設定 ★
    # -------------------------------------------------------------
    # 【研究室PC用設定】相対パスで指定 (image-anomaly-detection直下を想定)
    TEST_DIR = "data/bottle/test"
    
    # -------------------------------------------------------------
    # ★ 検証パラメータの設定 ★
    # -------------------------------------------------------------
    # 異常スコア算出に使う上位パッチ数 (ボトルのキズには130程度が経験的に良好)
    TOP_K = 130
    
    # ★ 重要: 閾値の設定 ★
    # ヒストグラム(plot_score_distribution)を見て、
    # 緑(正常)と赤(異常)の山が最もきれいに分かれる値をここに設定してください。
    # 最初は 0.5 程度で実行し、グラフを見て微調整するのが王道です。
    THRESHOLD = 0.5 
    
    # -------------------------------------------------------------
    # 検証プロセスの実行
    # -------------------------------------------------------------
    
    # 1. データセット全件評価 (推論とスコア算出)
    y_true, y_pred, scores, all_image_paths = evaluate_dataset(model, TEST_DIR, device, THRESHOLD, top_k=TOP_K)
    
    # 2. 定量評価 (混同行列、Accuracy, Recall) の描画
    plot_confusion_matrix(y_true, y_pred, THRESHOLD, top_k=TOP_K)
    
    # 3. 異常スコア分布 (ヒストグラム) の描画
    plot_score_distribution(y_true, scores, THRESHOLD, top_k=TOP_K)
    
    # 4. パターン別（TN, FP, FN, TP）の抽出と可視化
    # 予測結果を元に画像を4つのグループに分類
    print("\n--- パターン別の代表例を可視化します ---")
    tn_paths, fp_paths, fn_paths, tp_paths = [], [], [], []
    for true_lbl, pred_lbl, path in zip(y_true, y_pred, all_image_paths):
        if true_lbl == 0 and pred_lbl == 0: tn_paths.append(path)   # True Negative
        elif true_lbl == 0 and pred_lbl == 1: fp_paths.append(path) # False Positive (過剰検知)
        elif true_lbl == 1 and pred_lbl == 0: fn_paths.append(path) # False Negative (見逃し)
        elif true_lbl == 1 and pred_lbl == 1: tp_paths.append(path) # True Positive
            
    # 各グループから最大2枚をランダムにピックアップして可視化
    categories = {
        "【1. TN: 正常を「正常」と正しく判断】": tn_paths,
        "【2. FP: 正常を「異常」と誤って判断 (過剰検知)】": fp_paths,
        "【3. FN: 異常を「正常」と誤って判断 (見逃し)】": fn_paths,
        "【4. TP: 異常を「異常」と正しく判断】": tp_paths
    }
    
    for title, paths in categories.items():
        sample_size = min(2, len(paths)) # 最大2枚
        print(f"\n{title} : 該当 {len(paths)} 件中 {sample_size} 件を表示")
        
        if sample_size > 0:
            sampled_paths = random.sample(paths, sample_size)
            for img_path in sampled_paths:
                plot_single_anomaly(model, img_path, device, THRESHOLD, top_k=TOP_K)
        else:
            print("  -> このパターンに該当する画像はありません。")

    print("\nすべての検証プロセスが完了しました。")

In [ ]:
# =====================================================================
# 検証用コード: 浅いCNNモデル（CNN_Shallow3）100epアニーリング版 専用
# 異常スコア: SSIM Top-K (上位130パッチの平均誤差)
# =====================================================================
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score
import random
from skimage.metrics import structural_similarity as ssim

# 文字化け対策の強制適用
import japanize_matplotlib
japanize_matplotlib.japanize()

# =====================================================================
# 1. 前処理とモデルの定義 (CNN_Shallow3版)
# =====================================================================
# 学習時と同じ前処理を定義
transform = transforms.Compose([
    transforms.CenterCrop(820),
    transforms.Resize((128, 128), interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

# 検証対象のモデルクラス定義 (学習コードからコピー)
class VAE_CNN_Shallow3(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_CNN_Shallow3, self).__init__()
        
        # エンコーダ: 畳み込み層で空間情報を保ちながら圧縮 (128 -> 64 -> 32 -> 16)
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        # 64チャンネル * 16 * 16 = 16384次元
        self.fc_mu = nn.Linear(64 * 16 * 16, latent_dim)
        self.fc_logvar = nn.Linear(64 * 16 * 16, latent_dim)
        
        self.decoder_fc = nn.Linear(latent_dim, 64 * 16 * 16)
        
        # デコーダ: 逆畳み込み層で元の画像サイズへ復元
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)
        
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0) # ★安全装置
        
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 64, 16, 16) # 4階テンソルに戻す
        recon = self.decoder_conv(h_recon)
        
        return recon, mu, logvar

# =====================================================================
# 2. 評価と可視化の関数群 (SSIM Top-K仕様)
# =====================================================================
def evaluate_dataset(model, test_dir, device, threshold, top_k=130):
    """データセット全体を推論し、評価指標、スコア、ファイルパスを返す"""
    model.eval()
    y_true, y_pred, scores, all_image_paths = [], [], [], []
    
    test_dir = Path(test_dir)
    if not test_dir.exists():
        raise FileNotFoundError(f"テストディレクトリが見つかりません: {test_dir}\n研究室PCのデータパスを確認してください。")

    print(f"テストデータ全件の評価を開始します... (モデル: CNN_Shallow3, Top-K: {top_k})")
    
    with torch.no_grad():
        for folder in test_dir.iterdir():
            if not folder.is_dir():
                continue
                
            true_label = 0 if folder.name == "good" else 1
            
            img_paths = list(folder.glob("*.png"))
            if not img_paths:
                continue

            for img_path in img_paths:
                img = Image.open(img_path).convert('RGB')
                input_tensor = transform(img).unsqueeze(0).to(device)
                
                recon_tensor, _, _ = model(input_tensor)
                
                input_img = input_tensor.squeeze().cpu().numpy()
                recon_img = recon_tensor.squeeze().cpu().numpy()
                
                _, ssim_map = ssim(input_img, recon_img, data_range=1.0, win_size=11, full=True)
                anomaly_map = 1.0 - ssim_map
                
                sorted_anomaly = np.sort(anomaly_map.flatten())[::-1]
                top_k_score = np.mean(sorted_anomaly[:top_k])
                
                pred_label = 1 if top_k_score > threshold else 0
                
                y_true.append(true_label)
                y_pred.append(pred_label)
                scores.append(top_k_score)
                all_image_paths.append(img_path)
                
    print("評価が完了しました。")
    return np.array(y_true), np.array(y_pred), np.array(scores), all_image_paths

def plot_confusion_matrix(y_true, y_pred, threshold, top_k=130):
    """混同行列と定量評価指標(正解率・再現率)を描画する"""
    acc = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    
    print(f"\n--- 定量評価結果 (CNN_Shallow3) ---")
    print(f"設定閾値: {threshold}")
    print(f"正解率 (Accuracy): {acc * 100:.2f}%")
    print(f"再現率 (Recall)  : {recall * 100:.2f}%\n")
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['正常予測', '異常予測'],
                yticklabels=['実際の正常', '実際の異常'],
                annot_kws={"size": 16})
    
    plt.title(f"混同行列 (CNN_Shallow3 | SSIM Top-{top_k})\nAcc: {acc*100:.1f}% | Recall: {recall*100:.1f}%", fontsize=16)
    plt.xlabel('AIの予測', fontsize=14)
    plt.ylabel('実際の正解', fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_score_distribution(y_true, scores, threshold, top_k=130):
    """正常画像と異常画像の異常スコア分布(ヒストグラム)を描画する"""
    plt.figure(figsize=(10, 6))
    
    normal_scores = scores[y_true == 0]
    anomaly_scores = scores[y_true == 1]
    
    min_score = np.min(scores)
    max_score = np.max(scores)
    common_bins = np.linspace(min_score, max_score, 40)
    
    sns.histplot(normal_scores, bins=common_bins, color='green', alpha=0.5, label='正常 (Good)', kde=False)
    sns.histplot(anomaly_scores, bins=common_bins, color='red', alpha=0.5, label='異常 (Anomaly)', kde=False)
    
    plt.axvline(x=threshold, color='blue', linestyle='--', linewidth=2, label=f'現在の閾値 ({threshold})')
    
    plt.title(f"異常スコアの分布 (CNN_Shallow3 | SSIM Top-{top_k})", fontsize=16, fontweight='bold')
    plt.xlabel(f"異常スコア (上位{top_k}パッチの類似度誤差平均)", fontsize=14)
    plt.ylabel("画像の数 (Count)", fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def plot_single_anomaly(model, image_path, device, threshold, top_k=130):
    """1枚の画像を3パネルで可視化する"""
    model.eval()
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        recon_tensor, _, _ = model(input_tensor)
        
    input_img = input_tensor.squeeze().cpu().numpy()
    recon_img = recon_tensor.squeeze().cpu().numpy()
    
    _, ssim_map = ssim(input_img, recon_img, data_range=1.0, win_size=11, full=True)
    anomaly_map = 1.0 - ssim_map
    sorted_anomaly = np.sort(anomaly_map.flatten())[::-1]
    top_k_score = np.mean(sorted_anomaly[:top_k])
    
    is_anomaly = top_k_score > threshold
    judgment = "【異常 (Anomaly)】" if is_anomaly else "【正常 (Good)】"
    result_color = 'red' if is_anomaly else 'green'

    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    fig.suptitle(f"ファイル: {image_path.name} (実際の正解: {image_path.parent.name})\nSSIM Top-{top_k}スコア: {top_k_score:.4f} -> 判定: {judgment}", 
                 fontsize=16, fontweight='bold', color=result_color)
    
    axes[0].imshow(input_img, cmap='gray')
    axes[0].set_title("① 入力画像 (Gray)", fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(recon_img, cmap='gray')
    axes[1].set_title("② VAE復元画像", fontsize=14)
    axes[1].axis('off')
    
    vmax_val = np.max(anomaly_map) if np.max(anomaly_map) > 0.1 else 0.1
    im3 = axes[2].imshow(anomaly_map, cmap='jet', vmin=0, vmax=vmax_val)
    axes[2].set_title(f"③ 異常度マップ (1-SSIM)\n(赤色: 構造が復元できていない箇所)", fontsize=14)
    axes[2].axis('off')
    fig.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# =====================================================================
# 3. 実行ブロック (研究室PCでの実行用)
# =====================================================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")
    
    # -------------------------------------------------------------
    # ★ モデルと重みファイルの指定 ★
    # -------------------------------------------------------------
    model = VAE_CNN_Shallow3(latent_dim=64).to(device)
    
    # 学習コードで設定したファイル名に合わせる
    MODEL_PATH = "saved_models/vae_CNN_Shallow3_Annealing_ep100_beta50_128.pth"
    
    if not Path(MODEL_PATH).exists():
        raise FileNotFoundError(f"重みファイルが見つかりません: {MODEL_PATH}\n学習が完了しているか、パスが正しいか確認してください。")
    
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(f"CNN_Shallow3 の重みを読み込みました。\n")
    
    # -------------------------------------------------------------
    # ★ テストデータのパス設定 ★
    # -------------------------------------------------------------
    TEST_DIR = "data/bottle/test"
    
    # -------------------------------------------------------------
    # ★ 検証パラメータの設定 ★
    # -------------------------------------------------------------
    TOP_K = 130
    THRESHOLD = 0.5 
    
    # -------------------------------------------------------------
    # 検証プロセスの実行
    # -------------------------------------------------------------
    y_true, y_pred, scores, all_image_paths = evaluate_dataset(model, TEST_DIR, device, THRESHOLD, top_k=TOP_K)
    plot_confusion_matrix(y_true, y_pred, THRESHOLD, top_k=TOP_K)
    plot_score_distribution(y_true, scores, THRESHOLD, top_k=TOP_K)
    
    print("\n--- パターン別の代表例を可視化します ---")
    tn_paths, fp_paths, fn_paths, tp_paths = [], [], [], []
    for true_lbl, pred_lbl, path in zip(y_true, y_pred, all_image_paths):
        if true_lbl == 0 and pred_lbl == 0: tn_paths.append(path)
        elif true_lbl == 0 and pred_lbl == 1: fp_paths.append(path)
        elif true_lbl == 1 and pred_lbl == 0: fn_paths.append(path)
        elif true_lbl == 1 and pred_lbl == 1: tp_paths.append(path)
            
    categories = {
        "【1. TN: 正常を「正常」と正しく判断】": tn_paths,
        "【2. FP: 正常を「異常」と誤って判断 (過剰検知)】": fp_paths,
        "【3. FN: 異常を「正常」と誤って判断 (見逃し)】": fn_paths,
        "【4. TP: 異常を「異常」と正しく判断】": tp_paths
    }
    
    for title, paths in categories.items():
        sample_size = min(2, len(paths))
        print(f"\n{title} : 該当 {len(paths)} 件中 {sample_size} 件を表示")
        
        if sample_size > 0:
            sampled_paths = random.sample(paths, sample_size)
            for img_path in sampled_paths:
                plot_single_anomaly(model, img_path, device, THRESHOLD, top_k=TOP_K)
        else:
            print("  -> このパターンに該当する画像はありません。")

    print("\nすべての検証プロセスが完了しました。")

In [ ]:
# =====================================================================
# 検証用コード: 深いCNNモデル（CNN_Deep5）100epアニーリング版 専用
# 異常スコア: SSIM Top-K (上位130パッチの平均誤差)
# =====================================================================
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score
import random
from skimage.metrics import structural_similarity as ssim

# 文字化け対策の強制適用
import japanize_matplotlib
japanize_matplotlib.japanize()

# =====================================================================
# 1. 前処理とモデルの定義 (CNN_Deep5版)
# =====================================================================
# 学習時と同じ前処理を定義
transform = transforms.Compose([
    transforms.CenterCrop(820),
    transforms.Resize((128, 128), interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

# 検証対象のモデルクラス定義 (学習コードからコピー)
class VAE_CNN_Deep5(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_CNN_Deep5, self).__init__()
        
        # エンコーダ: 畳み込み層を5層重ねてさらに深く圧縮 (128 -> 64 -> 32 -> 16 -> 8 -> 4)
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # --- ここから追加した深い層 ---
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )
        
        # 最終層のサイズ: 256チャンネル * 4ピクセル * 4ピクセル = 4096次元
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)
        
        # デコーダ用の全結合層
        self.decoder_fc = nn.Linear(latent_dim, 256 * 4 * 4)
        
        # デコーダ: 逆畳み込み層を5層重ねて画像サイズを復元 (4 -> 8 -> 16 -> 32 -> 64 -> 128)
        self.decoder_conv = nn.Sequential(
            # --- ここから追加した深い層 ---
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # ------------------------------

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)
        
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        
        # ★重要: GPUでのKLD爆発を防ぐ安全装置
        logvar = torch.clamp(logvar, max=10.0)
        
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 256, 4, 4) # 4階テンソルに戻す（256ch, 4x4）
        recon = self.decoder_conv(h_recon)
        
        return recon, mu, logvar

# =====================================================================
# 2. 評価と可視化の関数群 (SSIM Top-K仕様)
# =====================================================================
def evaluate_dataset(model, test_dir, device, threshold, top_k=130):
    """データセット全体を推論し、評価指標、スコア、ファイルパスを返す"""
    model.eval()
    y_true, y_pred, scores, all_image_paths = [], [], [], []
    
    test_dir = Path(test_dir)
    if not test_dir.exists():
        raise FileNotFoundError(f"テストディレクトリが見つかりません: {test_dir}\n研究室PCのデータパスを確認してください。")

    print(f"テストデータ全件の評価を開始します... (モデル: CNN_Deep5, Top-K: {top_k})")
    
    with torch.no_grad():
        for folder in test_dir.iterdir():
            if not folder.is_dir():
                continue
                
            true_label = 0 if folder.name == "good" else 1
            
            img_paths = list(folder.glob("*.png"))
            if not img_paths:
                continue

            for img_path in img_paths:
                img = Image.open(img_path).convert('RGB')
                input_tensor = transform(img).unsqueeze(0).to(device)
                
                recon_tensor, _, _ = model(input_tensor)
                
                input_img = input_tensor.squeeze().cpu().numpy()
                recon_img = recon_tensor.squeeze().cpu().numpy()
                
                _, ssim_map = ssim(input_img, recon_img, data_range=1.0, win_size=11, full=True)
                anomaly_map = 1.0 - ssim_map
                
                sorted_anomaly = np.sort(anomaly_map.flatten())[::-1]
                top_k_score = np.mean(sorted_anomaly[:top_k])
                
                pred_label = 1 if top_k_score > threshold else 0
                
                y_true.append(true_label)
                y_pred.append(pred_label)
                scores.append(top_k_score)
                all_image_paths.append(img_path)
                
    print("評価が完了しました。")
    return np.array(y_true), np.array(y_pred), np.array(scores), all_image_paths

def plot_confusion_matrix(y_true, y_pred, threshold, top_k=130):
    """混同行列と定量評価指標(正解率・再現率)を描画する"""
    acc = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    
    print(f"\n--- 定量評価結果 (CNN_Deep5) ---")
    print(f"設定閾値: {threshold}")
    print(f"正解率 (Accuracy): {acc * 100:.2f}%")
    print(f"再現率 (Recall)  : {recall * 100:.2f}%\n")
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['正常予測', '異常予測'],
                yticklabels=['実際の正常', '実際の異常'],
                annot_kws={"size": 16})
    
    plt.title(f"混同行列 (CNN_Deep5 | SSIM Top-{top_k})\nAcc: {acc*100:.1f}% | Recall: {recall*100:.1f}%", fontsize=16)
    plt.xlabel('AIの予測', fontsize=14)
    plt.ylabel('実際の正解', fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_score_distribution(y_true, scores, threshold, top_k=130):
    """正常画像と異常画像の異常スコア分布(ヒストグラム)を描画する"""
    plt.figure(figsize=(10, 6))
    
    normal_scores = scores[y_true == 0]
    anomaly_scores = scores[y_true == 1]
    
    min_score = np.min(scores)
    max_score = np.max(scores)
    common_bins = np.linspace(min_score, max_score, 40)
    
    sns.histplot(normal_scores, bins=common_bins, color='green', alpha=0.5, label='正常 (Good)', kde=False)
    sns.histplot(anomaly_scores, bins=common_bins, color='red', alpha=0.5, label='異常 (Anomaly)', kde=False)
    
    plt.axvline(x=threshold, color='blue', linestyle='--', linewidth=2, label=f'現在の閾値 ({threshold})')
    
    plt.title(f"異常スコアの分布 (CNN_Deep5 | SSIM Top-{top_k})", fontsize=16, fontweight='bold')
    plt.xlabel(f"異常スコア (上位{top_k}パッチの類似度誤差平均)", fontsize=14)
    plt.ylabel("画像の数 (Count)", fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def plot_single_anomaly(model, image_path, device, threshold, top_k=130):
    """1枚の画像を3パネルで可視化する"""
    model.eval()
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        recon_tensor, _, _ = model(input_tensor)
        
    input_img = input_tensor.squeeze().cpu().numpy()
    recon_img = recon_tensor.squeeze().cpu().numpy()
    
    _, ssim_map = ssim(input_img, recon_img, data_range=1.0, win_size=11, full=True)
    anomaly_map = 1.0 - ssim_map
    sorted_anomaly = np.sort(anomaly_map.flatten())[::-1]
    top_k_score = np.mean(sorted_anomaly[:top_k])
    
    is_anomaly = top_k_score > threshold
    judgment = "【異常 (Anomaly)】" if is_anomaly else "【正常 (Good)】"
    result_color = 'red' if is_anomaly else 'green'

    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    fig.suptitle(f"ファイル: {image_path.name} (実際の正解: {image_path.parent.name})\nSSIM Top-{top_k}スコア: {top_k_score:.4f} -> 判定: {judgment}", 
                 fontsize=16, fontweight='bold', color=result_color)
    
    axes[0].imshow(input_img, cmap='gray')
    axes[0].set_title("① 入力画像 (Gray)", fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(recon_img, cmap='gray')
    axes[1].set_title("② VAE復元画像", fontsize=14)
    axes[1].axis('off')
    
    vmax_val = np.max(anomaly_map) if np.max(anomaly_map) > 0.1 else 0.1
    im3 = axes[2].imshow(anomaly_map, cmap='jet', vmin=0, vmax=vmax_val)
    axes[2].set_title(f"③ 異常度マップ (1-SSIM)\n(赤色: 構造が復元できていない箇所)", fontsize=14)
    axes[2].axis('off')
    fig.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# =====================================================================
# 3. 実行ブロック (研究室PCでの実行用)
# =====================================================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")
    
    # -------------------------------------------------------------
    # ★ モデルと重みファイルの指定 ★
    # -------------------------------------------------------------
    model = VAE_CNN_Deep5(latent_dim=64).to(device)
    
    # 学習コードで設定したファイル名に合わせる
    MODEL_PATH = "saved_models/vae_CNN_Deep5_Annealing_ep100_beta50_128.pth"
    
    if not Path(MODEL_PATH).exists():
        raise FileNotFoundError(f"重みファイルが見つかりません: {MODEL_PATH}\n学習が完了しているか、パスが正しいか確認してください。")
    
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(f"CNN_Deep5 の重みを読み込みました。\n")
    
    # -------------------------------------------------------------
    # ★ テストデータのパス設定 ★
    # -------------------------------------------------------------
    TEST_DIR = "data/bottle/test"
    
    # -------------------------------------------------------------
    # ★ 検証パラメータの設定 ★
    # -------------------------------------------------------------
    TOP_K = 130
    THRESHOLD = 0.5 
    
    # -------------------------------------------------------------
    # 検証プロセスの実行
    # -------------------------------------------------------------
    y_true, y_pred, scores, all_image_paths = evaluate_dataset(model, TEST_DIR, device, THRESHOLD, top_k=TOP_K)
    plot_confusion_matrix(y_true, y_pred, THRESHOLD, top_k=TOP_K)
    plot_score_distribution(y_true, scores, THRESHOLD, top_k=TOP_K)
    
    print("\n--- パターン別の代表例を可視化します ---")
    tn_paths, fp_paths, fn_paths, tp_paths = [], [], [], []
    for true_lbl, pred_lbl, path in zip(y_true, y_pred, all_image_paths):
        if true_lbl == 0 and pred_lbl == 0: tn_paths.append(path)
        elif true_lbl == 0 and pred_lbl == 1: fp_paths.append(path)
        elif true_lbl == 1 and pred_lbl == 0: fn_paths.append(path)
        elif true_lbl == 1 and pred_lbl == 1: tp_paths.append(path)
            
    categories = {
        "【1. TN: 正常を「正常」と正しく判断】": tn_paths,
        "【2. FP: 正常を「異常」と誤って判断 (過剰検知)】": fp_paths,
        "【3. FN: 異常を「正常」と誤って判断 (見逃し)】": fn_paths,
        "【4. TP: 異常を「異常」と正しく判断】": tp_paths
    }
    
    for title, paths in categories.items():
        sample_size = min(2, len(paths))
        print(f"\n{title} : 該当 {len(paths)} 件中 {sample_size} 件を表示")
        
        if sample_size > 0:
            sampled_paths = random.sample(paths, sample_size)
            for img_path in sampled_paths:
                plot_single_anomaly(model, img_path, device, THRESHOLD, top_k=TOP_K)
        else:
            print("  -> このパターンに該当する画像はありません。")

    print("\nすべての検証プロセスが完了しました。")

In [ ]:
# =====================================================================
# 検証用コード: ハイブリッドモデル（Hybrid_CNN5_FC）100epアニーリング版 専用
# 異常スコア: SSIM Top-K (上位130パッチの平均誤差)
# =====================================================================
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score
import random
from skimage.metrics import structural_similarity as ssim

# 文字化け対策の強制適用
import japanize_matplotlib
japanize_matplotlib.japanize()

# =====================================================================
# 1. 前処理とモデルの定義 (Hybrid_CNN5_FC版)
# =====================================================================
# 学習時と同じ前処理を定義
transform = transforms.Compose([
    transforms.CenterCrop(820),
    transforms.Resize((128, 128), interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

# 検証対象のモデルクラス定義 (学習コードからコピー)
class VAE_Hybrid_CNN_FC(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_Hybrid_CNN_FC, self).__init__()
        
        # -------------------------------------------------------------
        # 【エンコーダ前半】 CNN_Deep5（空間圧縮: 128x128 -> 4x4）
        # -------------------------------------------------------------
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),
            
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU()
        )
        
        # -------------------------------------------------------------
        # 【エンコーダ後半】 FC_Only（段階的圧縮: 4096次元 -> 128次元）
        # -------------------------------------------------------------
        self.encoder_fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )
        
        # 潜在空間へのマッピング（128次元 -> 64次元）
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # -------------------------------------------------------------
        # 【デコーダ前半】 FC_Only（段階的復元: 64次元 -> 4096次元）
        # -------------------------------------------------------------
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            nn.Linear(1024, 256 * 4 * 4), nn.ReLU()
        )
        
        # -------------------------------------------------------------
        # 【デコーダ後半】 CNN_Deep5（空間復元: 4x4 -> 128x128）
        # -------------------------------------------------------------
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),
            
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # 1. CNNによる特徴抽出
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)
        
        # 2. FCによる段階的圧縮
        h = self.encoder_fc(h)
        
        # 3. 潜在変数の計算と安全装置（クランプ）
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        
        # 4. FCによる段階的復元
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 256, 4, 4) # 4階テンソルに戻す
        
        # 5. CNNによる画像復元
        recon = self.decoder_conv(h_recon)
        
        return recon, mu, logvar

# =====================================================================
# 2. 評価と可視化の関数群 (SSIM Top-K仕様)
# =====================================================================
def evaluate_dataset(model, test_dir, device, threshold, top_k=130):
    """データセット全体を推論し、評価指標、スコア、ファイルパスを返す"""
    model.eval()
    y_true, y_pred, scores, all_image_paths = [], [], [], []
    
    test_dir = Path(test_dir)
    if not test_dir.exists():
        raise FileNotFoundError(f"テストディレクトリが見つかりません: {test_dir}\n研究室PCのデータパスを確認してください。")

    print(f"テストデータ全件の評価を開始します... (モデル: Hybrid_CNN5_FC, Top-K: {top_k})")
    
    with torch.no_grad():
        for folder in test_dir.iterdir():
            if not folder.is_dir():
                continue
                
            true_label = 0 if folder.name == "good" else 1
            
            img_paths = list(folder.glob("*.png"))
            if not img_paths:
                continue

            for img_path in img_paths:
                img = Image.open(img_path).convert('RGB')
                input_tensor = transform(img).unsqueeze(0).to(device)
                
                recon_tensor, _, _ = model(input_tensor)
                
                input_img = input_tensor.squeeze().cpu().numpy()
                recon_img = recon_tensor.squeeze().cpu().numpy()
                
                _, ssim_map = ssim(input_img, recon_img, data_range=1.0, win_size=11, full=True)
                anomaly_map = 1.0 - ssim_map
                
                sorted_anomaly = np.sort(anomaly_map.flatten())[::-1]
                top_k_score = np.mean(sorted_anomaly[:top_k])
                
                pred_label = 1 if top_k_score > threshold else 0
                
                y_true.append(true_label)
                y_pred.append(pred_label)
                scores.append(top_k_score)
                all_image_paths.append(img_path)
                
    print("評価が完了しました。")
    return np.array(y_true), np.array(y_pred), np.array(scores), all_image_paths

def plot_confusion_matrix(y_true, y_pred, threshold, top_k=130):
    """混同行列と定量評価指標(正解率・再現率)を描画する"""
    acc = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    
    print(f"\n--- 定量評価結果 (Hybrid_CNN5_FC) ---")
    print(f"設定閾値: {threshold}")
    print(f"正解率 (Accuracy): {acc * 100:.2f}%")
    print(f"再現率 (Recall)  : {recall * 100:.2f}%\n")
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['正常予測', '異常予測'],
                yticklabels=['実際の正常', '実際の異常'],
                annot_kws={"size": 16})
    
    plt.title(f"混同行列 (Hybrid_CNN5_FC | SSIM Top-{top_k})\nAcc: {acc*100:.1f}% | Recall: {recall*100:.1f}%", fontsize=16)
    plt.xlabel('AIの予測', fontsize=14)
    plt.ylabel('実際の正解', fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_score_distribution(y_true, scores, threshold, top_k=130):
    """正常画像と異常画像の異常スコア分布(ヒストグラム)を描画する"""
    plt.figure(figsize=(10, 6))
    
    normal_scores = scores[y_true == 0]
    anomaly_scores = scores[y_true == 1]
    
    min_score = np.min(scores)
    max_score = np.max(scores)
    common_bins = np.linspace(min_score, max_score, 40)
    
    sns.histplot(normal_scores, bins=common_bins, color='green', alpha=0.5, label='正常 (Good)', kde=False)
    sns.histplot(anomaly_scores, bins=common_bins, color='red', alpha=0.5, label='異常 (Anomaly)', kde=False)
    
    plt.axvline(x=threshold, color='blue', linestyle='--', linewidth=2, label=f'現在の閾値 ({threshold})')
    
    plt.title(f"異常スコアの分布 (Hybrid_CNN5_FC | SSIM Top-{top_k})", fontsize=16, fontweight='bold')
    plt.xlabel(f"異常スコア (上位{top_k}パッチの類似度誤差平均)", fontsize=14)
    plt.ylabel("画像の数 (Count)", fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def plot_single_anomaly(model, image_path, device, threshold, top_k=130):
    """1枚の画像を3パネルで可視化する"""
    model.eval()
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        recon_tensor, _, _ = model(input_tensor)
        
    input_img = input_tensor.squeeze().cpu().numpy()
    recon_img = recon_tensor.squeeze().cpu().numpy()
    
    _, ssim_map = ssim(input_img, recon_img, data_range=1.0, win_size=11, full=True)
    anomaly_map = 1.0 - ssim_map
    sorted_anomaly = np.sort(anomaly_map.flatten())[::-1]
    top_k_score = np.mean(sorted_anomaly[:top_k])
    
    is_anomaly = top_k_score > threshold
    judgment = "【異常 (Anomaly)】" if is_anomaly else "【正常 (Good)】"
    result_color = 'red' if is_anomaly else 'green'

    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    fig.suptitle(f"ファイル: {image_path.name} (実際の正解: {image_path.parent.name})\nSSIM Top-{top_k}スコア: {top_k_score:.4f} -> 判定: {judgment}", 
                 fontsize=16, fontweight='bold', color=result_color)
    
    axes[0].imshow(input_img, cmap='gray')
    axes[0].set_title("① 入力画像 (Gray)", fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(recon_img, cmap='gray')
    axes[1].set_title("② VAE復元画像", fontsize=14)
    axes[1].axis('off')
    
    vmax_val = np.max(anomaly_map) if np.max(anomaly_map) > 0.1 else 0.1
    im3 = axes[2].imshow(anomaly_map, cmap='jet', vmin=0, vmax=vmax_val)
    axes[2].set_title(f"③ 異常度マップ (1-SSIM)\n(赤色: 構造が復元できていない箇所)", fontsize=14)
    axes[2].axis('off')
    fig.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# =====================================================================
# 3. 実行ブロック (研究室PCでの実行用)
# =====================================================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")
    
    # -------------------------------------------------------------
    # ★ モデルと重みファイルの指定 ★
    # -------------------------------------------------------------
    model = VAE_Hybrid_CNN_FC(latent_dim=64).to(device)
    
    # 学習コードで設定したファイル名に合わせる
    MODEL_PATH = "saved_models/vae_Hybrid_CNN5_FC_Annealing_ep100_beta50_128.pth"
    
    if not Path(MODEL_PATH).exists():
        raise FileNotFoundError(f"重みファイルが見つかりません: {MODEL_PATH}\n学習が完了しているか、パスが正しいか確認してください。")
    
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(f"Hybrid_CNN5_FC の重みを読み込みました。\n")
    
    # -------------------------------------------------------------
    # ★ テストデータのパス設定 ★
    # -------------------------------------------------------------
    TEST_DIR = "data/bottle/test"
    
    # -------------------------------------------------------------
    # ★ 検証パラメータの設定 ★
    # -------------------------------------------------------------
    TOP_K = 130
    THRESHOLD = 0.5 
    
    # -------------------------------------------------------------
    # 検証プロセスの実行
    # -------------------------------------------------------------
    y_true, y_pred, scores, all_image_paths = evaluate_dataset(model, TEST_DIR, device, THRESHOLD, top_k=TOP_K)
    plot_confusion_matrix(y_true, y_pred, THRESHOLD, top_k=TOP_K)
    plot_score_distribution(y_true, scores, THRESHOLD, top_k=TOP_K)
    
    print("\n--- パターン別の代表例を可視化します ---")
    tn_paths, fp_paths, fn_paths, tp_paths = [], [], [], []
    for true_lbl, pred_lbl, path in zip(y_true, y_pred, all_image_paths):
        if true_lbl == 0 and pred_lbl == 0: tn_paths.append(path)
        elif true_lbl == 0 and pred_lbl == 1: fp_paths.append(path)
        elif true_lbl == 1 and pred_lbl == 0: fn_paths.append(path)
        elif true_lbl == 1 and pred_lbl == 1: tp_paths.append(path)
            
    categories = {
        "【1. TN: 正常を「正常」と正しく判断】": tn_paths,
        "【2. FP: 正常を「異常」と誤って判断 (過剰検知)】": fp_paths,
        "【3. FN: 異常を「正常」と誤って判断 (見逃し)】": fn_paths,
        "【4. TP: 異常を「異常」と正しく判断】": tp_paths
    }
    
    for title, paths in categories.items():
        sample_size = min(2, len(paths))
        print(f"\n{title} : 該当 {len(paths)} 件中 {sample_size} 件を表示")
        
        if sample_size > 0:
            sampled_paths = random.sample(paths, sample_size)
            for img_path in sampled_paths:
                plot_single_anomaly(model, img_path, device, THRESHOLD, top_k=TOP_K)
        else:
            print("  -> このパターンに該当する画像はありません。")

    print("\nすべての検証プロセスが完了しました。")